In [2]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

In [3]:
load_dotenv()


True

In [4]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

# Get project directory
BASE_DIR = Path.cwd().parent

# Document directory
DOCUMENTS_DIR = BASE_DIR / "documents"

# Load all PDF documents
documents = []

for pdf_file in DOCUMENTS_DIR.glob("*.pdf"):

    loader = PyPDFLoader(str(pdf_file))

    pages = loader.load()

    documents.extend(pages)

    print(f"Loaded: {pdf_file.name}")
    print(f"Pages: {len(pages)}")

print("\nTotal pages loaded:", len(documents))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_12300\1994258924.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded: CNC_Error_Code_Reference.pdf
Pages: 4
Loaded: CNC_Machine_Operation_Manual (1).pdf
Pages: 3
Loaded: CNC_Preventive_Maintenance_Manual.pdf
Pages: 4
Loaded: CNC_Safety_Manual.pdf
Pages: 4
Loaded: CNC_Troubleshooting_Guide.pdf
Pages: 4

Total pages loaded: 19


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

In [7]:
chunks = text_splitter.split_documents(documents)

print("Total documents:", len(documents))
print("Total chunks:", len(chunks))

Total documents: 19
Total chunks: 59


In [8]:
for i, chunk in enumerate(chunks[:3]):

    print(f"\n========== CHUNK {i+1} ==========")

    print("Content:")
    print(chunk.page_content)

    print("\nMetadata:")
    print(chunk.metadata)


========== CHUNK 1 ==========
Content:
GENERAL CNC REFERENCE  Verify against machine-builder documentation
Page 1
 CNC MACHINE OPERATIONS & CARE
 CNC Error Code Reference
 Example alarm categories  meanings  alarm-handling procedures
DOCUMENT CONTROL
DETAIL
Document
CNC_Error_Code_Reference.pdf
Edition
1.0 — General reference edition
Prepared
11 November 2026
Applicability
General CNC machining centres, lathes and routers; adapt locally
Important: This is an educational, generic reference. CNC machines differ substantially by builder, model, control,
options, tooling and site rules. The machine-specific operator, maintenance, electrical and safety manuals take
precedence. Do not use example values, procedures or alarm descriptions as substitutes for approved machine
documentation.

Metadata:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-11T04:40:56+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-11-11T04:40:56

In [9]:
from langchain_ollama import OllamaEmbeddings

In [10]:
embedding_model = OllamaEmbeddings(
    model="all-minilm",
    base_url="http://127.0.0.1:11434"
)

In [11]:
sample_text = "CNC spindle overheating can occur due to excessive load or lubrication problems."

vector = embedding_model.embed_query(sample_text)

print("Embedding generated successfully!")
print("Vector length:", len(vector))
print("First 10 values:", vector[:10])

Embedding generated successfully!
Vector length: 384
First 10 values: [-0.0674169, -0.1237242, 0.020481532, -0.0012323787, -0.036980025, -0.07211298, -0.06521161, 0.016479198, 0.0076648793, 0.054243013]


In [12]:
from langchain_community.vectorstores import FAISS

In [13]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS vector database created successfully!")

FAISS vector database created successfully!


In [14]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [15]:
from langchain_groq import ChatGroq

In [16]:
api_key = os.getenv("GROQ_API_KEY")

In [17]:
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.3,
    api_key=api_key
)

In [18]:
llm.invoke("what is generative ai?").content

'**Generative AI** (Artificial Intelligence) is a type of artificial intelligence capable of creating new content, such as text, images, audio, video, and even code. Unlike traditional AI, which is typically used for analyzing data or making predictions based on existing patterns, generative AI can produce original outputs that mimic human creativity.\n\n### How It Works\nMost modern generative AI systems are built on **deep learning** models, particularly:\n- **Large Language Models (LLMs)**: For text generation (e.g., GPT-4, Claude, Llama).\n- **Diffusion Models**: For image and video generation (e.g., DALL·E, Midjourney, Stable Diffusion).\n- **Transformers**: A neural network architecture that processes sequences of data (like words or pixels) by focusing on relationships between elements.\n\nThese models are trained on massive datasets of text, images, or other media. During training, they learn the statistical patterns and structures of the data. When given a prompt (e.g., “Write

In [19]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
You are a Manufacturing Maintenance Knowledge Assistant.

Your job is to answer the user's question using ONLY the
information provided in the context.

Rules:
- Do not use outside knowledge.
- Do not guess or make up information.
- If the context contains the answer, explain it clearly.
- If the context does not contain enough information to answer
  the question, respond exactly:
  "The information is not available in the provided documents."
- Keep the answer concise and easy to understand.
- Use bullet points or numbered lists when appropriate.
- Do not mention document names, page numbers, metadata,
  retrieved chunks, or sources.

Context:
{context}

Question:
{question}

Answer:
""")

In [20]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableLambda

In [21]:
format_docs = RunnableLambda(
    lambda docs: "\n\n".join(
        doc.page_content for doc in docs
    )
)

In [22]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [23]:
question = "what is use of CNC milling machine?"

response = rag_chain.invoke(question)

print(response)

The use of a CNC milling machine is to remove material using a rotating cutting tool. It is commonly used to produce:
*   Slots
*   Pockets
*   Faces
*   Complex shapes
